# Ask 2 — Chunking: Cut the Pile into Pieces

The chunk is the unit of retrieval: what gets scored, found, and handed to
the model. This notebook cuts the pile three ways and measures which cuts
keep answers intact. Fully offline.

In [ ]:
# The pile: documents from the (fictional) Jefferson High School.
# Real enough to search, small enough to read whole.
PILE = {
 "handbook_academics": """S4.1 Grading scale. A 90-100, B 80-89, C 70-79, D 60-69.
Semester grades weight exams at 30 percent.
S4.2 Exam Retake Policy. This policy applies to final exams only. Students
receive one retake per semester, requested within ten school days. The
higher score stands.
S4.3 Grade appeals. Appeals go to the department head in writing within
fifteen school days of the posted grade.
S4.5 Late work. Assignments lose 10 percent per school day late, to a
maximum of 50 percent. Teachers may grant extensions for documented
emergencies.""",
 "handbook_schedule": """S2.0 Bell schedule. Regular days run eight periods,
8:15 AM to 3:20 PM.
S2.1 Wednesday schedule. Dismissal at 1:30 PM every Wednesday for staff
development.
S2.4 Late arrival. Students arriving after 8:30 AM sign in at the main
office with a note.""",
 "handbook_trips": """S5.1 Field trips require a signed permission form
submitted five school days in advance.
S5.2 Trip costs above 20 dollars qualify for the student activity fund.
S5.4 Chaperones must be approved district volunteers.""",
 "handbook_athletics": """S6.2 Eligibility. Athletes must hold a C average
during their season. Freshmen may try out for varsity teams.
S6.3 Petitions. A varsity roster spot for a freshman requires a coach's
petition to the athletic director.""",
 "robotics_minutes": """Robotics club meets Tuesdays in room 214. Regional
trip is April 18; bring your signed permission form by April 10. Dues are
15 dollars for the year.""",
 "clubs_list": """Active clubs: robotics (Tuesdays), debate (Thursdays),
art collective (Fridays), chess (lunch, library). Sign-up forms at the
student office.""",
 "bus_routes": """Routes 12 and 15 serve the north side. Final pickup at
4:45 PM outside door C. Activity buses run Tuesday and Thursday only.""",
 "cafeteria": """Lunch periods run 11:10, 11:55, and 12:40. Breakfast is
served from 7:40 AM. Menus post monthly on the food services page.""",
}
print(f"{len(PILE)} documents, {sum(len(t) for t in PILE.values())} characters total")

## Three knives

In [ ]:
def chunk_fixed(pile, size=120):
    """Every `size` characters, meaning be damned."""
    chunks = []
    for doc, text in pile.items():
        flat = " ".join(text.split())
        for i in range(0, len(flat), size):
            chunks.append({"doc": doc, "section": f"piece{i//size}", "text": flat[i:i+size]})
    return chunks

def chunk_whole(pile):
    """One chunk per document."""
    return [{"doc": doc, "section": "whole", "text": " ".join(text.split())} for doc, text in pile.items()]
def chunk_by_section(pile, overlap_sentences=1):
    """Cut on the S-section seams; carry a sentence of overlap across cuts."""
    chunks = []
    for doc, text in pile.items():
        parts, current, header = [], [], None
        for line in text.splitlines():
            if line.strip().startswith("S") and len(line) > 2 and line.strip()[1].isdigit():
                if current:
                    parts.append((header, " ".join(current)))
                header, current = line.strip().split()[0].rstrip("."), [line]
            else:
                current.append(line)
        if current:
            parts.append((header, " ".join(current)))
        for i, (header, body) in enumerate(parts):
            text_out = body
            if overlap_sentences and i > 0:
                prev_tail = parts[i-1][1].split(". ")[-1]
                text_out = prev_tail + " ... " + body
            chunks.append({"doc": doc, "section": header or doc, "text": " ".join(text_out.split())})
    return chunks

CHUNKS = chunk_by_section(PILE)  # the section knife
print(f"{len(CHUNKS)} chunks")
for c in CHUNKS[:3]:
    print(f"  [{c['doc']} {c['section']}] {c['text'][:70]}...")

## The measurement: does the answer survive the knife?

Five questions, each answered by a known span of text. A chunking "keeps"
an answer if some single chunk contains the whole span — because retrieval
returns chunks, and a half-answer chunk is true and useless.

In [ ]:
ANSWERS = [
    ("How many final retakes do I get?",
     "one retake per semester, requested within ten school days"),
    ("When is Wednesday dismissal?", "Dismissal at 1:30 PM every Wednesday"),
    ("When are permission forms due?", "permission form submitted five school days in advance"),
    ("Can freshmen make varsity?", "requires a coach's petition"),
    ("What's the late-work penalty?", "lose 10 percent per school day late"),
]

def keeps(chunks, span):
    span_flat = " ".join(span.split())
    return any(span_flat in c["text"] for c in chunks)

cuts = {"tiny (120 chars)": chunk_fixed(PILE), "whole document": chunk_whole(PILE),
        "by section + overlap": chunk_by_section(PILE)}
print(f"{'cut':22} {'chunks':>6}  answers kept intact")
for name, chunks in cuts.items():
    kept = sum(keeps(chunks, span) for q, span in ANSWERS)
    print(f"{name:22} {len(chunks):6d}  {kept} of {len(ANSWERS)}")

assert sum(keeps(chunk_by_section(PILE), s) for q, s in ANSWERS) == len(ANSWERS)
assert sum(keeps(chunk_fixed(PILE), s) for q, s in ANSWERS) < len(ANSWERS), \
       "the tiny knife should cut through at least one answer"

## Where the tiny knife cut

Whole-document chunks keep every span too — the failure they cause is
burial (one relevant line inside forty), which shows up as bad retrieval
scores in lesson 3, not as a lost span here. Both numbers matter.

In [ ]:
tiny = chunk_fixed(PILE)
for q, span in ANSWERS:
    if not keeps(tiny, span):
        print("severed:", repr(span))
        pieces = [c for c in tiny if span.split()[0] in c["text"] or span.split()[-1] in c["text"]]
        for p in pieces[:2]:
            print(f"   [{p['doc']} {p['section']}] ...{p['text'][-60:]}")

## Try it

1. Raise the fixed size until it keeps all five answers. What did the
   chunk count fall to — and what did each chunk turn into?
2. Set `overlap_sentences=0` in the section chunker and find a question
   the overlap was protecting.
3. **Build turn-in:** your own pile chunked; counts, size range, two
   well-cut chunks, one hurt one, and your fix.